In [27]:
from google.colab import drive
drive.mount('/content/drive')

import pickle

with open('/content/drive/MyDrive/aqua spatial/enugu_data.pkl', 'rb') as f:
    data = pickle.load(f)

enugu = data['enugu']
wp_gdf = data['wp_gdf']
enugu_stats = data['enugu_stats']
lga_stats = data['lga_stats']

print(f"Loaded from Drive")
print(f"   LGAs: {len(enugu)}")
print(f"   Water points: {len(wp_gdf)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded from Drive
   LGAs: 17
   Water points: 292


 Reminder: Resuing Lagos--uhi-project for this project

In [28]:
import ee
import geemap
import pandas as pd
import numpy as np
import time

# In Colab, this opens a browser auth popup
ee.Authenticate()
ee.Initialize(project='lagos-uhi-analysis')

print("GEE initialized — lagos-uhi-analysis")
print(f"   geemap version: {geemap.__version__}")

GEE initialized — lagos-uhi-analysis
   geemap version: 0.37.2


Defining Enugu's Area of Interest (AOI) from LGA centroids

In [29]:
# LGA centroids as feature collection
enugu_centroids = enugu.copy()
# Replace these two lines in Cell 3:
enugu_centroids['centroid_lon'] = enugu.geometry.to_crs(epsg=32632).centroid.to_crs(epsg=4326).x
enugu_centroids['centroid_lat'] = enugu.geometry.to_crs(epsg=32632).centroid.to_crs(epsg=4326).y

# Convert Enugu boundary to EE geometry
enugu_bbox = ee.Geometry.Rectangle([6.906, 5.919, 7.865, 7.113])

# Build EE FeatureCollection from centroids
features = []
for _, row in enugu_centroids.iterrows():
    feat = ee.Feature(
        ee.Geometry.Point([row['centroid_lon'], row['centroid_lat']]),
        {'lga_name': row['NAME_2']}
    )
    features.append(feat)


lga_fc = ee.FeatureCollection(features)
print(f" Feature collection built: 17 LGA centroids")

 Feature collection built: 17 LGA centroids


Extracting All four real Satelite datasets

In [30]:
# --- 1. NDVI from Landsat 8 (2023 dry season mean) ---
landsat = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(enugu_bbox)
    .filterDate('2023-11-01', '2024-02-28')  # Dry season
    .filter(ee.Filter.lt('CLOUD_COVER', 20))
    .map(lambda img: img.normalizedDifference(['SR_B5', 'SR_B4'])
         .rename('NDVI')
         .copyProperties(img, ['system:time_start']))
    .mean()
    .clip(enugu_bbox))

# --- 2. SMAP Soil Moisture (2023 annual mean) ---
smap = (ee.ImageCollection('NASA/SMAP/SPL4SMGP/007')
    .filterDate('2023-01-01', '2023-12-31')
    .select('sm_surface')
    .mean()
    .clip(enugu_bbox))

# --- 3. SRTM Elevation ---
srtm = ee.Image('USGS/SRTMGL1_003').clip(enugu_bbox)
elevation = srtm.select('elevation')
slope = ee.Terrain.slope(srtm)

# --- 4. Land Surface Temperature MODIS (dry season) ---
lst = (ee.ImageCollection('MODIS/061/MOD11A1')
    .filterDate('2023-11-01', '2024-02-28')
    .select('LST_Day_1km')
    .mean()
    .multiply(0.02).subtract(273.15)  # Convert to Celsius
    .clip(enugu_bbox))

print("All 4 image collections defined")
print("   → NDVI (vegetation/groundwater proxy)")
print("   → Soil Moisture (SMAP)")
print("   → Elevation + Slope (SRTM)")
print("   → Land Surface Temperature (MODIS)")

All 4 image collections defined
   → NDVI (vegetation/groundwater proxy)
   → Soil Moisture (SMAP)
   → Elevation + Slope (SRTM)
   → Land Surface Temperature (MODIS)


## Sampling all Four datasets in the Centroids

Basically, we are asking GEE servers to go through the last cell and bring back the values of the NDVI, SMAP, SRTM and MODIS for Enugu's 17 LGA Centers.

The Original approach was that GEE computes everything and tries to send it all at once but then the memory explodes.

So I Fixed the approach by
GEE computing band 1 to band 5 separately so it writes CSV to Drive...


We load all 5 CSVs and merged locally.

In [31]:
def export_band_to_drive(image, band_name, lga_fc, scale=10000):
    """Sample a single band at LGA centroids and export to Drive"""
    sampled = image.rename(band_name).sampleRegions(
        collection=lga_fc,
        scale=scale,
        geometries=False
    )
    task = ee.batch.Export.table.toDrive(
        collection=sampled,
        description=f'enugu_{band_name}',
        folder='aqua spatial',       # That's my folder's name
        fileNamePrefix=f'enugu_{band_name}',
        fileFormat='CSV'
    )
    task.start()
    return task

# Exporting each band separately which avoids composite memory overflow as explained in the markdown above
print("Starting export tasks...")

tasks = {
    'ndvi':         export_band_to_drive(landsat, 'ndvi', lga_fc, scale=10000),
    'soil_moisture':export_band_to_drive(smap, 'soil_moisture', lga_fc, scale=10000),
    'elevation':    export_band_to_drive(elevation, 'elevation', lga_fc, scale=10000),
    'slope':        export_band_to_drive(slope, 'slope', lga_fc, scale=10000),
    'lst_celsius':  export_band_to_drive(lst, 'lst_celsius', lga_fc, scale=10000),
}

print(f"{len(tasks)} export tasks submitted to GEE")
print("\nTask statuses:")
for name, task in tasks.items():
    print(f"   {name}: {task.status()['state']}")

print("\nCheck GEE Tasks tab — exports usually complete in 2–5 minutes")
print("   Then run Cell 6 to load all CSVs from Drive")

Starting export tasks...
5 export tasks submitted to GEE

Task statuses:
   ndvi: RUNNING
   soil_moisture: RUNNING
   elevation: READY
   slope: READY
   lst_celsius: READY

Check GEE Tasks tab — exports usually complete in 2–5 minutes
   Then run Cell 6 to load all CSVs from Drive


### Loading the exported CSVs and Merging them into One DataFrame

We collected the answers:
17 rows × 6 columns of actual satellite readings. NDVI from Landsat 8. Soil moisture from SMAP. Elevation from SRTM. Slope computed from terrain. Temperature from MODIS. All real. All from space.

In [32]:
# Checking the task statues as it takes a little bit of time
print("Checking task statuses...")
for name, task in tasks.items():
    status = task.status()['state']
    print(f"   {name}: {status}")

print("\nIf all show COMPLETED, run the merge below.")
print("If still RUNNING, wait 2 mins and recheck.\n")

# Load and merge all band CSVs
base_path = '/content/drive/MyDrive/aqua spatial/'

bands = ['ndvi', 'soil_moisture', 'elevation', 'slope', 'lst_celsius']
dfs = []

for band in bands:
    path = f'{base_path}enugu_{band}.csv'
    df = pd.read_csv(path)[['lga_name', band]]
    dfs.append(df)
    print(f"Loaded: {band} — {len(df)} rows")

# Merge all on lga_name
from functools import reduce
gee_df = reduce(lambda l, r: pd.merge(l, r, on='lga_name'), dfs)

print(f"\nFinal GEE feature table: {gee_df.shape}")
print(gee_df.round(4))

Checking task statuses...
   ndvi: COMPLETED
   soil_moisture: COMPLETED
   elevation: READY
   slope: RUNNING
   lst_celsius: RUNNING

If all show COMPLETED, run the merge below.
If still RUNNING, wait 2 mins and recheck.

Loaded: ndvi — 17 rows
Loaded: soil_moisture — 17 rows
Loaded: elevation — 17 rows
Loaded: slope — 17 rows
Loaded: lst_celsius — 17 rows

Final GEE feature table: (17, 6)
         lga_name    ndvi  soil_moisture  elevation   slope  lst_celsius
0          Aninri  0.2316         0.3459         57  0.0930      29.6848
1            Awgu  0.2477         0.3653        329  0.3349      28.5386
2       EnuguEast  0.1670         0.2282        171  0.4031      30.5385
3      EnuguNorth  0.1115         0.2605        152  0.5083      31.0762
4      EnuguSouth  0.1115         0.2605        152  0.5083      31.0762
5          Ezeagu  0.2477         0.2136         47  0.1715      29.9565
6      Igbo-Etiti  0.2628         0.2629        459  0.3423      28.0728
7   Igbo-ezeNorth  0.